# Telecom Egypt RAG Colab Full Pipeline

This notebook builds data and indexes for the Telecom Egypt Intelligent Assistant RAG project. It is for scraping, processing, embedding, Qdrant indexing, and snapshot export. It does not run or modify the Streamlit app.

## 1. Introduction

Run the cells one stage at a time. The default mode is a limited test (`MAX_PAGES_PER_SECTION = 5`) so you can validate Drive outputs before launching the full build.

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORKSPACE_DIR = '/content/drive/MyDrive/telecom_egypt_rag_colab'
REPO_DIR = '/content/telecom-egypt-rag'
print('Workspace:', WORKSPACE_DIR)
print('Repo:', REPO_DIR)

## 3. Configure Pipeline

In [ ]:
# Limited test mode. Use this first.
MAX_PAGES_PER_SECTION = 5
CONCURRENCY = 2
DELAY_SECONDS = 1.0

# Full mode. Switch RUN_FULL_MODE to True after the test mode looks good.
RUN_FULL_MODE = False

OVERWRITE_PROCESSED = False
FORCE_REFETCH = False
FORCE_REEMBED = False
DYNAMIC = False
ENABLE_MOBILE = False

EMBEDDING_MODEL = 'qwen3-embedding:4b'
COLLECTION_NAME = 'telecom_all_sources_v2'
QDRANT_IMAGE = 'qdrant/qdrant:v1.14.0'

## 4. Clone Repo

In [ ]:
# Set REPO_URL to your GitHub URL before running in a fresh Colab runtime.
REPO_URL = ''  # example: 'https://github.com/<owner>/telecom-egypt-rag.git'

import os
import subprocess
from pathlib import Path

repo_path = Path(REPO_DIR)
if repo_path.exists():
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
elif REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    raise ValueError('Set REPO_URL or upload/clone the project into REPO_DIR first.')

## 5. Install Dependencies

In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'qdrant-client', 'tqdm', 'pyyaml', 'requests', 'beautifulsoup4', 'lxml', 'scrapling[fetchers]>=0.4.9'], check=True)

## 6. Prepare Runner

In [ ]:
import sys
from pathlib import Path

import os
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.colab_full_pipeline import build_config, parse_args

base_args = [
    '--workspace-dir', WORKSPACE_DIR,
    '--repo-dir', REPO_DIR,
    '--max-pages-per-section', str(MAX_PAGES_PER_SECTION),
    '--concurrency', str(CONCURRENCY),
    '--delay-seconds', str(DELAY_SECONDS),
    '--overwrite-processed', str(OVERWRITE_PROCESSED),
    '--force-refetch', str(FORCE_REFETCH),
    '--force-reembed', str(FORCE_REEMBED),
    '--dynamic', str(DYNAMIC),
    '--collection-name', COLLECTION_NAME,
    '--embedding-model', EMBEDDING_MODEL,
    '--qdrant-image', QDRANT_IMAGE,
    '--enable-mobile', str(ENABLE_MOBILE),
]
if RUN_FULL_MODE:
    base_args.append('--full')

config = build_config(parse_args(base_args))
print(config)

## 7. Create Drive Workspace

In [ ]:
from scripts.colab_full_pipeline import ensure_workspace

ensure_workspace(config)
!find {WORKSPACE_DIR} -maxdepth 2 -type d | sort

## 8. Scrape Sections

In [ ]:
import asyncio
from scripts.colab_full_pipeline import scrape_sections

scrape_manifest = await scrape_sections(config)
scrape_manifest

## 9. Review Scraping Outputs

In [ ]:
!find {WORKSPACE_DIR}/raw_html -type f | head -20
!find {WORKSPACE_DIR}/extracted_records -type f | sort
!find {WORKSPACE_DIR}/processed -type f | sort
!find {WORKSPACE_DIR}/quality_reports -type f | sort

## 10. Build KB and Chunks

In [ ]:
from scripts.colab_full_pipeline import build_unified_kb, build_chunks

kb_outputs = build_unified_kb(config)
chunk_outputs = build_chunks(config)
kb_outputs, chunk_outputs

## 11. Build BM25

In [ ]:
from scripts.colab_full_pipeline import build_bm25

bm25_outputs = build_bm25(config)
bm25_outputs

## 12. Install Ollama

In [ ]:
from scripts.colab_full_pipeline import install_and_start_ollama

install_and_start_ollama(config)
!ollama list

## 13. Start Qdrant

In [ ]:
from scripts.colab_full_pipeline import start_qdrant_docker

start_qdrant_docker(config)
!curl -s http://localhost:6333/collections | python -m json.tool

## 14. Generate Embeddings

In [ ]:
from scripts.colab_full_pipeline import generate_embeddings_and_upsert

embedding_manifest = generate_embeddings_and_upsert(config)
embedding_manifest

## 15. Export Snapshot

In [ ]:
from scripts.colab_full_pipeline import create_qdrant_snapshot

snapshot_manifest = create_qdrant_snapshot(config)
snapshot_manifest

## 16. Download Artifacts

In [ ]:
from google.colab import files
from pathlib import Path

snapshot_path = snapshot_manifest.get('snapshot_path')
if snapshot_path and Path(snapshot_path).exists():
    files.download(snapshot_path)
else:
    print('Snapshot file is saved in Drive. Manifest:', snapshot_manifest)

embedded_points = Path(WORKSPACE_DIR) / 'embedded_points' / 'embedded_points_v2.jsonl.gz'
if embedded_points.exists():
    print('Embedded backup:', embedded_points)

## 17. Final Report

In [ ]:
from scripts.colab_full_pipeline import write_final_report

final_report = write_final_report(config)
final_report

## 18. Restore Locally

In [ ]:
from scripts.colab_full_pipeline import print_restore_instructions

print_restore_instructions(config)